# Exploratory Data Analysis

### Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")

### Load the Data

In [ ]:
df_clean = pd.read_csv("clean_data.csv")

### Overview

In [ ]:
display(df_clean.head())
display(df_clean.describe(include="all").T)

### Distributions

In [ ]:
numeric_cols = df_clean.select_dtypes(
    include=np.number
).columns.tolist()

categorical_cols = df_clean.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()


if numeric_cols:

    n_cols = 3
    n_rows = int(np.ceil(len(numeric_cols) / n_cols))

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(16, 4 * n_rows)
    )

    axes = np.array(axes).reshape(-1)

    for i, col in enumerate(numeric_cols):

        sns.histplot(
            data=df_clean,
            x=col,
            kde=True,
            ax=axes[i]
        )

        axes[i].set_title(f"Distribution of {col}")

    for j in range(len(numeric_cols), len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()

### Boxplot Distributions

In [ ]:
if numeric_cols:

    n_cols = 3
    n_rows = int(np.ceil(len(numeric_cols) / n_cols))

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(16, 4 * n_rows)
    )

    axes = np.array(axes).reshape(-1)

    for i, col in enumerate(numeric_cols):

        sns.boxplot(
            data=df_clean,
            x=col,
            ax=axes[i]
        )

        axes[i].set_title(f"Boxplot: {col}")

    for j in range(len(numeric_cols), len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()

### Categorical Distributions (Barplots)

In [ ]:
for col in categorical_cols:

    unique_count = df_clean[col].nunique()

    # Avoid generating unreadable charts for high-cardinality columns
    if unique_count <= 20:

        plt.figure(figsize=(10, 5))

        order = df_clean[col].value_counts().index

        sns.countplot(
            data=df_clean,
            y=col,
            order=order
        )

        plt.title(f"Distribution of {col}")
        plt.tight_layout()
        plt.show()

    else:
        print(f"\n{col}: {unique_count} unique categories")
        display(df_clean[col].value_counts().head(10))

### Correlation Matrix

In [ ]:
if len(numeric_cols) >= 2:

    correlation_matrix = df_clean[numeric_cols].corr(
        method="pearson"
    )

    plt.figure(figsize=(12, 9))

    sns.heatmap(
        correlation_matrix,
        annot=True,
        cmap="coolwarm",
        center=0,
        fmt=".2f",
        linewidths=0.5
    )

    plt.title("Correlation Matrix")
    plt.tight_layout()
    plt.show()

    # Identify strongest correlations
    corr_pairs = (
        correlation_matrix
        .where(
            np.triu(
                np.ones(correlation_matrix.shape),
                k=1
            ).astype(bool)
        )
        .stack()
        .reset_index()
    )

    corr_pairs.columns = [
        "variable_1", "variable_2", "correlation"
    ]

    corr_pairs["absolute_correlation"] = (
        corr_pairs["correlation"].abs()
    )

    display(
        corr_pairs.sort_values(
            "absolute_correlation",
            ascending=False
        ).head(15)
    )

### Bivariate Analyses

In [ ]:
# Automatically examine relationships between numerical variables

if len(numeric_cols) >= 2:

    top_numeric = numeric_cols[:6]

    sns.pairplot(
        df_clean[top_numeric].dropna().sample(
            min(1000, len(df_clean.dropna(subset=top_numeric))),
            random_state=42
        ),
        corner=True,
        diag_kind="hist"
    )

    plt.show()



In [ ]:
# Compare numerical metrics across categorical groups
for cat_col in categorical_cols:

    if df_clean[cat_col].nunique() <= 15:

        for num_col in numeric_cols[:5]:

            grouped = (
                df_clean.groupby(cat_col, observed=True)[num_col]
                .agg(["count", "mean", "median", "std"])
                .sort_values("mean", ascending=False)
            )

            print(f"\n{num_col} by {cat_col}:")
            display(grouped)